In [1]:
# COLAB SETUP

%cd /content
!rm -rf /content/proto-tsrl
!git clone https://github.com/haiyan-wang/proto-tsrl.git /content/proto-tsrl
%cd /content/proto-tsrl

from google.colab import drive, runtime
drive.mount('/content/drive')

import sys
import os

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(project_root)

/content
Cloning into '/content/proto-tsrl'...
remote: Enumerating objects: 763, done.
remote: Counting objects: 100% (370/370), done.
remote: Compressing objects: 100% (287/287), done.
remote: Total 763 (delta 218), reused 182 (delta 83), pack-reused 393 (from 1)
Receiving objects: 100% (763/763), 34.74 MiB | 33.85 MiB/s, done.
Resolving deltas: 100% (431/431), done.
/content/proto-tsrl
Mounted at /content/drive
/content/proto-tsrl


In [2]:
import functools

import numpy as np

from sklearn.model_selection import StratifiedShuffleSplit

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

from src.models import PPGModel
from src.utils.sampling_utils import *
from src.utils.training_utils import *
from src.experiments.ppg.ppg_data_utils import *

ModuleNotFoundError: No module named 'src.experiments.ppg.ppg_model'

In [ ]:
# SETTINGS

SEED = 42

# device
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(DEVICE)

# data
INCLUDE_CLEAN_DATA = True
INCLUDE_SEMINOISY_DATA = True
INCLUDE_NOISY_DATA = False
TRAIN_SET_SIZE = int(1e5)
SFT_SET_SIZE = 5000

# dataloaders
BATCH_SIZE = 256
VAL_RATIO = 0.05
N_WORKERS = 4

# contrastive sampling
# PPG data has length 800, rescaled to [0, 1]
COLLATE_FN_KWARGS = {
    'min_len': 100,
    'max_len': 300,
    'num_neg': 5,
    'min_overlap': 0.3,
    'min_var': 1e-4,
    'max_var': 1e-3
}

# schedule
N_EPOCHS_STAGE1 = 20
N_EPOCHS_STAGE2 = 10
N_EPOCHS_STAGE3 = 10
TOTAL_EPOCHS = N_EPOCHS_STAGE1 + N_EPOCHS_STAGE2 + N_EPOCHS_STAGE3

# logging
CHECKPOINT_EPOCHS = [*range(1, TOTAL_EPOCHS + 1)] # 1-indexed
SAVE_DIR = "/content/drive/MyDrive/Duke/Senior Year/Thesis/experiments/ppg/checkpoints"

# architecture
MASK_PROB = 0.2
MID_WEIGHT = 0.5
PROTO_NEG_MARGIN = 0.1
PROTO_DIVERSITY_THRESHOLD = 0.2
LAMBDA_PROTO = 1.0
TEMPERATURE = 1.0
LAMBDA_REPR = 1e-3
GRADIENT_CLIP = 1.0

# parameter settings and optimizers
REP_DIMS = [10, 50, 100]
MODELS = []
OPTIMIZER_DICTS = []
CKPT_PATHS = []
for dim in REP_DIMS:
    model = PPGModel(representation_dimension = dim, mask_probability = MASK_PROB)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(DEVICE)
    MODELS.append(model)

    ckpt_path = f"{SAVE_DIR}/dim{dim}"
    CKPT_PATHS.append(ckpt_path)

    opt_dict = {
    "stage1": torch.optim.Adam(model.parameters(), lr = 1e-3, weight_decay = 1e-5),
    "stage2": torch.optim.Adam(model.parameters(), lr = 1e-3, weight_decay = 1e-5),
    "stage3": torch.optim.Adam(model.parameters(), lr = 1e-4, weight_decay = 1e-5),
    }
    OPTIMIZER_DICTS.append(opt_dict)

SCHEDULER = None

cuda:0


In [ ]:
# LOAD DATA

X_train, y_train = load_data(
    file_path = '/content/drive/MyDrive/Duke/Senior Year/Thesis/ppg_data',
    clean = INCLUDE_CLEAN_DATA,
    seminoisy = INCLUDE_SEMINOISY_DATA,
    noisy = INCLUDE_NOISY_DATA,
    dataset = 'train',
    return_labels = True
)

indices = np.random.default_rng(SEED).permutation(X_train.shape[0])
X_train, y_train = X_train[indices][:TRAIN_SET_SIZE], y_train[indices][:TRAIN_SET_SIZE]

print(f'X_train shape: {X_train.shape}')
print(f'Train set positive samples: {np.sum(y_train)}')

dl_train, dl_val = make_dataloaders(
    X_train,
    y_train,
    batch_size = BATCH_SIZE,
    val_ratio = VAL_RATIO,
    seed = SEED,
    num_workers = N_WORKERS,
    collate_fn_kwargs = COLLATE_FN_KWARGS
)

X_train shape: (100000, 800, 1)
Train set positive samples: 36118


In [ ]:
### TRAINING

for i, dim in enumerate(REP_DIMS):
    print(f'TRAINING MODEL WITH REPRESENTATION DIMENSION {dim}')
    history = run_training(
        model = MODELS[i],
        train_loader = dl_train,
        val_loader = dl_val,
        device = DEVICE,
        optimizer_dict = OPTIMIZER_DICTS[i],
        epochs_stage1 = N_EPOCHS_STAGE1,
        epochs_stage2 = N_EPOCHS_STAGE2,
        epochs_stage3 = N_EPOCHS_STAGE3,
        scheduler_dict = SCHEDULER,
        mid_weight = MID_WEIGHT,
        proto_neg_margin = PROTO_NEG_MARGIN,
        proto_diversity_threshold = PROTO_DIVERSITY_THRESHOLD,
        lambda_proto = LAMBDA_PROTO,
        temperature = TEMPERATURE,
        lambda_repr = LAMBDA_REPR,
        grad_clip_norm = GRADIENT_CLIP,
        checkpoint_path = CKPT_PATHS[i],
        checkpoint_epochs = CHECKPOINT_EPOCHS,
        collector_fn = None,
        use_amp = True
    )

TRAINING MODEL WITH REPRESENTATION DIMENSION 10
Training for 40 epochs.

=== stage1 (20 epochs) ===
[stage1] epoch 1/20 | global 1/40
  train loss: 2.092453 | val loss: 1.336205
Saved checkpoint at epoch 1
[stage1] epoch 2/20 | global 2/40
  train loss: 0.988151 | val loss: 0.788343
Saved checkpoint at epoch 2
[stage1] epoch 3/20 | global 3/40
  train loss: 0.709163 | val loss: 0.675282
Saved checkpoint at epoch 3
[stage1] epoch 4/20 | global 4/40
  train loss: 0.654487 | val loss: 0.617743
Saved checkpoint at epoch 4
[stage1] epoch 5/20 | global 5/40
  train loss: 0.617952 | val loss: 0.605705
Saved checkpoint at epoch 5
[stage1] epoch 6/20 | global 6/40
  train loss: 0.586255 | val loss: 0.578755
Saved checkpoint at epoch 6
[stage1] epoch 7/20 | global 7/40
  train loss: 0.569157 | val loss: 0.561700
Saved checkpoint at epoch 7
[stage1] epoch 8/20 | global 8/40
  train loss: 0.557527 | val loss: 0.553366
Saved checkpoint at epoch 8
[stage1] epoch 9/20 | global 9/40
  train loss: 0.55

In [ ]:
runtime.unassign()